# DSA Interview Prep Notebook — Agoda / FAANG Edition
### Complete guide: patterns · templates · tricks · edge cases · practice problems
---
**How to use this notebook:**
- Each section = one pattern or data structure
- Read the *Mental Model* first, then study the *Template Code*
- Check *Common Mistakes* before each practice session
- Use *Practice Problems* links to drill on LeetCode / HackerRank

> **Difficulty tags:** 🟢 Easy · 🟡 Medium · 🔴 Hard


---
# 1. Arrays
---

## 1.1 Two Pointers

### Mental Model
Two pointers are indices that move toward each other (or in the same direction) to avoid a nested loop.
Use when: sorted array + pair/triplet condition, or in-place manipulation.

**Decision tree:**
- Opposite ends → pair sum, container water, palindrome check
- Same direction → remove duplicates, fast/slow, partition

### Complexity
| Operation | Time | Space |
|-----------|------|-------|
| Standard  | O(n) | O(1)  |
| After sort| O(n log n) | O(1) |

### Common Mistakes
- ❌ Forgetting to sort before applying two pointers on unsorted input
- ❌ Off-by-one: `while lo < hi` vs `while lo <= hi` (use `<` for pairs, `<=` for single element)
- ❌ Not handling duplicates — skip duplicate values after processing

### Edge Cases
- Empty array or length 1
- All elements identical
- No valid pair exists → return -1 or []
- Negative numbers (sort handles them fine)

### Practice Problems
| # | Problem | Difficulty | Pattern |
|---|---------|-----------|---------|
| LC 167 | Two Sum II | 🟢 | opposite ends |
| LC 15  | 3Sum | 🟡 | sort + two ptr |
| LC 11  | Container With Most Water | 🟡 | opposite ends |
| LC 42  | Trapping Rain Water | 🔴 | prefix max trick |
| LC 75  | Sort Colors (Dutch Flag) | 🟡 | 3-pointer partition |


In [ ]:
# ── Two Pointers Templates ──────────────────────────────────────────────

# Template 1: Opposite ends — Two Sum on sorted array
def two_sum_sorted(nums, target):
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        s = nums[lo] + nums[hi]
        if s == target:   return [lo, hi]
        elif s < target:  lo += 1
        else:             hi -= 1
    return []

# Template 2: 3Sum — sort + fix one + two pointer
def three_sum(nums):
    nums.sort()
    res = []
    for i in range(len(nums) - 2):
        if i > 0 and nums[i] == nums[i-1]: continue   # skip duplicates
        lo, hi = i + 1, len(nums) - 1
        while lo < hi:
            s = nums[i] + nums[lo] + nums[hi]
            if s == 0:
                res.append([nums[i], nums[lo], nums[hi]])
                while lo < hi and nums[lo] == nums[lo+1]: lo += 1  # skip dups
                while lo < hi and nums[hi] == nums[hi-1]: hi -= 1
                lo += 1; hi -= 1
            elif s < 0: lo += 1
            else:       hi -= 1
    return res

# Template 3: Container with most water
def max_water(height):
    lo, hi = 0, len(height) - 1
    best = 0
    while lo < hi:
        best = max(best, min(height[lo], height[hi]) * (hi - lo))
        if height[lo] < height[hi]: lo += 1   # move shorter side inward
        else:                        hi -= 1
    return best

# Template 4: Dutch National Flag (sort 0s, 1s, 2s in-place)
def sort_colors(nums):
    lo, mid, hi = 0, 0, len(nums) - 1
    while mid <= hi:
        if   nums[mid] == 0: nums[lo], nums[mid] = nums[mid], nums[lo]; lo += 1; mid += 1
        elif nums[mid] == 1: mid += 1
        else:                nums[mid], nums[hi] = nums[hi], nums[mid]; hi -= 1

# ── Quick Tests ──────────────────────────────────────────────────────────
print(two_sum_sorted([2,7,11,15], 9))        # [0, 1]
print(three_sum([-1, 0, 1, 2, -1, -4]))      # [[-1,-1,2],[-1,0,1]]
print(max_water([1,8,6,2,5,4,8,3,7]))        # 49
a = [2,0,2,1,1,0]; sort_colors(a); print(a) # [0,0,1,1,2,2]


## 1.2 Sliding Window

### Mental Model
Maintain a window [lo, hi] over the array. Expand `hi` greedily; shrink `lo` when a constraint is violated.

**Fixed window:** both pointers move together — use for "exactly k elements" problems.
**Variable window:** `hi` drives, `lo` reacts — use for "longest/shortest X" problems.

### Key Template
```
lo = 0
for hi in range(n):
    # include nums[hi] in window
    while window_is_invalid:
        # exclude nums[lo] from window
        lo += 1
    # update answer from current valid window
```

### Complexity: O(n) time, O(1)–O(k) space

### Common Mistakes
- ❌ Using a nested loop instead of the two-pointer amortised approach → O(n²)
- ❌ Fixed-window: forgetting to subtract the element leaving the window
- ❌ Not initialising the first window for fixed-size problems
- ❌ HashMap not cleaned up when shrinking window

### Practice Problems
| # | Problem | Difficulty | Type |
|---|---------|-----------|------|
| LC 3   | Longest Substring Without Repeating | 🟡 | variable, max |
| LC 209 | Minimum Size Subarray Sum | 🟡 | variable, min |
| LC 239 | Sliding Window Maximum | 🔴 | fixed + deque |
| LC 76  | Minimum Window Substring | 🔴 | variable, freq |
| LC 567 | Permutation in String | 🟡 | fixed, freq |


In [ ]:
# ── Sliding Window Templates ────────────────────────────────────────────

# Variable window: longest substring without repeating characters
def length_of_longest_substring(s):
    seen = {}   # char → last seen index
    lo = best = 0
    for hi, c in enumerate(s):
        if c in seen and seen[c] >= lo:
            lo = seen[c] + 1        # jump past previous occurrence
        seen[c] = hi
        best = max(best, hi - lo + 1)
    return best

# Variable window: minimum size subarray sum >= target
def min_subarray_len(target, nums):
    lo = window_sum = 0
    best = float('inf')
    for hi in range(len(nums)):
        window_sum += nums[hi]
        while window_sum >= target:
            best = min(best, hi - lo + 1)
            window_sum -= nums[lo]; lo += 1
    return best if best != float('inf') else 0

# Fixed window: sliding window maximum (uses monotonic deque)
from collections import deque
def sliding_window_max(nums, k):
    dq = deque()  # stores indices, decreasing values
    res = []
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] <= x: dq.pop()
        dq.append(i)
        if dq[0] == i - k: dq.popleft()    # out of window
        if i >= k - 1: res.append(nums[dq[0]])
    return res

# Variable window: permutation in string (fixed-size anagram check)
def check_inclusion(s1, s2):
    from collections import Counter
    need = Counter(s1)
    have = Counter()
    formed = 0; required = len(need)
    lo = 0
    for hi, c in enumerate(s2):
        have[c] += 1
        if c in need and have[c] == need[c]: formed += 1
        while formed == required:
            if hi - lo + 1 == len(s1): return True
            have[s2[lo]] -= 1
            if s2[lo] in need and have[s2[lo]] < need[s2[lo]]: formed -= 1
            lo += 1
    return False

# ── Tests ────────────────────────────────────────────────────────────────
print(length_of_longest_substring("abcabcbb"))   # 3
print(min_subarray_len(7, [2,3,1,2,4,3]))        # 2
print(sliding_window_max([1,3,-1,-3,5,3,6,7],3)) # [3,3,5,5,6,7]
print(check_inclusion("ab", "eidbaooo"))          # True


## 1.3 Prefix Sum

### Mental Model
Precompute `prefix[i] = sum(arr[0..i-1])` so that `sum(l, r) = prefix[r+1] - prefix[l]` in O(1).

**2D prefix sum:** `pre[i][j] = sum of rectangle from (0,0) to (i-1,j-1)`

### Complexity: O(n) build, O(1) query

### Common Mistakes
- ❌ Off-by-one: use `prefix[0] = 0` sentinel → `prefix[i+1] = prefix[i] + arr[i]`
- ❌ Subarray sum = K: forgetting `freq = {0: 1}` base case → misses subarrays starting at index 0
- ❌ Modifying the original array instead of using a separate prefix array

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 560 | Subarray Sum Equals K | 🟡 |
| LC 238 | Product of Array Except Self | 🟡 |
| LC 304 | Range Sum Query 2D | 🟡 |
| LC 724 | Find Pivot Index | 🟢 |
| LC 1248 | Count Nice Subarrays | 🟡 |


In [ ]:
# ── Prefix Sum Templates ────────────────────────────────────────────────

# Build prefix sum array
def build_prefix(arr):
    prefix = [0] * (len(arr) + 1)
    for i, v in enumerate(arr):
        prefix[i+1] = prefix[i] + v
    return prefix   # sum(l,r) = prefix[r+1] - prefix[l]

# Subarray sum = K  (prefix + hashmap)
def subarray_sum_k(nums, k):
    count = 0; prefix = 0
    freq = {0: 1}               # base: empty subarray sums to 0
    for x in nums:
        prefix += x
        count += freq.get(prefix - k, 0)
        freq[prefix] = freq.get(prefix, 0) + 1
    return count

# Pivot index: left_sum == right_sum
def find_pivot_index(nums):
    total = sum(nums); left = 0
    for i, x in enumerate(nums):
        if left == total - left - x: return i
        left += x
    return -1

# Product of array except self (left + right prefix products)
def product_except_self(nums):
    n = len(nums)
    out = [1] * n
    # left pass
    for i in range(1, n): out[i] = out[i-1] * nums[i-1]
    # right pass (running right product)
    right = 1
    for i in range(n-1, -1, -1):
        out[i] *= right
        right *= nums[i]
    return out

# 2D prefix sum
def build_2d_prefix(matrix):
    nr, nc = len(matrix), len(matrix[0])
    pre = [[0]*(nc+1) for _ in range(nr+1)]
    for i in range(1, nr+1):
        for j in range(1, nc+1):
            pre[i][j] = (matrix[i-1][j-1] + pre[i-1][j]
                         + pre[i][j-1] - pre[i-1][j-1])
    # query sum of (r1,c1)..(r2,c2) (0-indexed):
    # pre[r2+1][c2+1] - pre[r1][c2+1] - pre[r2+1][c1] + pre[r1][c1]
    return pre

# ── Tests ────────────────────────────────────────────────────────────────
print(subarray_sum_k([1,1,1], 2))           # 2
print(find_pivot_index([1,7,3,6,5,6]))      # 3
print(product_except_self([1,2,3,4]))        # [24,12,8,6]


## 1.4 Sorting Patterns

### When to Think "Sort First"
- Finding pairs/triplets with a property
- Greedy problems (sort by key attribute)
- Merge intervals
- Counting inversions

### Complexity
| Algorithm | Time avg | Time worst | Space |
|-----------|----------|-----------|-------|
| Python sort (Timsort) | O(n log n) | O(n log n) | O(n) |
| Custom key sort | O(n log n) | — | O(1) key |

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 56  | Merge Intervals | 🟡 |
| LC 57  | Insert Interval | 🟡 |
| LC 252 | Meeting Rooms | 🟢 |
| LC 435 | Non-overlapping Intervals | 🟡 |
| LC 179 | Largest Number | 🟡 |


In [ ]:
# ── Sorting Patterns ────────────────────────────────────────────────────

# Merge intervals
def merge_intervals(intervals):
    intervals.sort(key=lambda x: x[0])
    merged = [intervals[0]]
    for start, end in intervals[1:]:
        if start <= merged[-1][1]:          # overlapping
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged

# Non-overlapping intervals (min removals to make non-overlapping)
def erase_overlap_intervals(intervals):
    intervals.sort(key=lambda x: x[1])     # sort by END time (greedy)
    removed = 0; end = float('-inf')
    for s, e in intervals:
        if s >= end: end = e                # keep this interval
        else:        removed += 1           # remove (it overlaps)
    return removed

# Largest number from array of ints
def largest_number(nums):
    from functools import cmp_to_key
    def cmp(a, b):
        if a+b > b+a: return -1
        elif a+b < b+a: return 1
        return 0
    nums = [str(n) for n in nums]
    nums.sort(key=cmp_to_key(cmp))
    return '0' if nums[0]=='0' else ''.join(nums)

# ── Tests ────────────────────────────────────────────────────────────────
print(merge_intervals([[1,3],[2,6],[8,10],[15,18]]))    # [[1,6],[8,10],[15,18]]
print(erase_overlap_intervals([[1,2],[2,3],[3,4],[1,3]])) # 1
print(largest_number([10,2]))                             # "210"


## 1.5 Subarrays & Kadane's Algorithm

### Mental Model — Kadane's
At each index, either extend the previous subarray or start a new one:
`dp[i] = max(nums[i], dp[i-1] + nums[i])`
Track the global max separately.

### Complexity: O(n) time, O(1) space

### Variants
- Max subarray sum → Kadane's
- Max subarray product → track both max and min (negatives flip sign)
- Circular subarray → max(standard_kadane, total_sum - min_kadane)

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 53  | Maximum Subarray | 🟢 |
| LC 152 | Maximum Product Subarray | 🟡 |
| LC 918 | Max Sum Circular Subarray | 🟡 |
| LC 325 | Maximum Size Subarray Sum = k | 🟡 |
| LC 523 | Continuous Subarray Sum | 🟡 |


In [ ]:
# ── Kadane's Algorithm & Variants ───────────────────────────────────────

# Classic Kadane — max subarray sum
def max_subarray(nums):
    cur = best = nums[0]
    for x in nums[1:]:
        cur = max(x, cur + x)
        best = max(best, cur)
    return best

# Max product subarray
def max_product(nums):
    max_p = min_p = best = nums[0]
    for x in nums[1:]:
        candidates = (x, max_p * x, min_p * x)
        max_p = max(candidates)
        min_p = min(candidates)
        best = max(best, max_p)
    return best

# Circular subarray max sum
def max_circular_subarray(nums):
    def kadane(arr):
        cur = best = arr[0]
        for x in arr[1:]:
            cur = max(x, cur + x); best = max(best, cur)
        return best
    max_sum = kadane(nums)
    total = sum(nums)
    # invert sign and find min subarray sum
    inv = [-x for x in nums]
    min_circular = total + kadane(inv)   # total - min_subarray
    # if all negative, min_circular = 0 (empty wrap) — invalid
    return max(max_sum, min_circular) if min_circular != 0 else max_sum

# Continuous subarray sum divisible by k (prefix + mod)
def check_subarray_sum(nums, k):
    # sum(i..j) % k == 0  iff  prefix[j] % k == prefix[i-1] % k
    seen = {0: -1}; prefix = 0
    for i, x in enumerate(nums):
        prefix = (prefix + x) % k
        if prefix in seen:
            if i - seen[prefix] >= 2: return True   # subarray len >= 2
        else:
            seen[prefix] = i
    return False

# ── Tests ────────────────────────────────────────────────────────────────
print(max_subarray([-2,1,-3,4,-1,2,1,-5,4]))   # 6
print(max_product([2,3,-2,4]))                  # 6
print(max_circular_subarray([1,-2,3,-2]))       # 3
print(check_subarray_sum([23,2,4,6,7], 6))      # True


---
# 2. Stacks & Queues
---

## 2.1 Monotonic Stack

### Mental Model
A stack that maintains elements in monotonically increasing or decreasing order.
Push each element; pop elements that violate the monotonic property.

**When to use:** "Next greater/smaller element", "Largest rectangle", "Trapping rain water".

| Direction | Stack order | Use for |
|-----------|------------|---------|
| Increasing | bottom→top | Next Greater Element |
| Decreasing | bottom→top | Next Smaller Element |

### Complexity: O(n) — each element pushed/popped at most once

### Common Mistakes
- ❌ Popping from stack before checking if it's empty
- ❌ Using the wrong comparison (> vs >=) for strict vs non-strict
- ❌ Forgetting to handle remaining elements in stack after the loop

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 739 | Daily Temperatures | 🟡 |
| LC 84  | Largest Rectangle in Histogram | 🔴 |
| LC 85  | Maximal Rectangle | 🔴 |
| LC 496 | Next Greater Element I | 🟢 |
| LC 42  | Trapping Rain Water | 🔴 |


In [ ]:
# ── Monotonic Stack Templates ────────────────────────────────────────────

# Next Greater Element (monotonic decreasing stack)
def next_greater_element(nums):
    n = len(nums)
    res = [-1] * n
    stack = []  # indices
    for i in range(n):
        while stack and nums[stack[-1]] < nums[i]:
            res[stack.pop()] = nums[i]
        stack.append(i)
    return res

# Daily temperatures
def daily_temperatures(temps):
    res = [0] * len(temps)
    stack = []  # indices
    for i, t in enumerate(temps):
        while stack and temps[stack[-1]] < t:
            j = stack.pop()
            res[j] = i - j
        stack.append(i)
    return res

# Largest rectangle in histogram
def largest_rect_histogram(heights):
    heights = [0] + heights + [0]   # sentinel 0s at both ends
    stack = [0]; max_area = 0
    for i in range(1, len(heights)):
        while heights[stack[-1]] > heights[i]:
            h = heights[stack.pop()]
            w = i - stack[-1] - 1
            max_area = max(max_area, h * w)
        stack.append(i)
    return max_area

# Trapping rain water (monotonic stack approach)
def trap_water(height):
    stack = []; water = 0
    for i, h in enumerate(height):
        while stack and height[stack[-1]] < h:
            bottom = height[stack.pop()]
            if not stack: break
            left = stack[-1]
            width = i - left - 1
            bounded = min(h, height[left]) - bottom
            water += bounded * width
        stack.append(i)
    return water

# ── Tests ────────────────────────────────────────────────────────────────
print(next_greater_element([2,1,2,4,3]))          # [4,2,4,-1,-1]
print(daily_temperatures([73,74,75,71,69,72,76,73])) # [1,1,4,2,1,1,0,0]
print(largest_rect_histogram([2,1,5,6,2,3]))      # 10
print(trap_water([0,1,0,2,1,0,1,3,2,1,2,1]))     # 6


## 2.2 Min/Max Stack & Stack Design Problems

### Mental Model
Augment a regular stack with O(1) getMin/getMax by storing auxiliary state alongside each element.

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 155  | Min Stack | 🟡 |
| LC 716  | Max Stack | 🟡 |
| LC 20   | Valid Parentheses | 🟢 |
| LC 32   | Longest Valid Parentheses | 🔴 |
| LC 1047 | Remove All Adjacent Duplicates | 🟢 |


In [ ]:
# ── Stack Design & Parentheses ───────────────────────────────────────────

# Min Stack — O(1) push/pop/getMin
class MinStack:
    def __init__(self):
        self.stack = []     # (val, current_min)
    def push(self, val):
        cur_min = min(val, self.stack[-1][1]) if self.stack else val
        self.stack.append((val, cur_min))
    def pop(self):   self.stack.pop()
    def top(self):   return self.stack[-1][0]
    def getMin(self):return self.stack[-1][1]

# Valid Parentheses
def is_valid_parens(s):
    match = {')':'(', ']':'[', '}':'{'}
    stack = []
    for c in s:
        if c in '([{': stack.append(c)
        elif not stack or stack[-1] != match[c]: return False
        else: stack.pop()
    return not stack

# Longest Valid Parentheses
def longest_valid_parens(s):
    stack = [-1]    # base index sentinel
    best = 0
    for i, c in enumerate(s):
        if c == '(':
            stack.append(i)
        else:
            stack.pop()
            if not stack: stack.append(i)   # new base
            else:         best = max(best, i - stack[-1])
    return best

# Remove All Adjacent Duplicates
def remove_duplicates(s):
    stack = []
    for c in s:
        if stack and stack[-1] == c: stack.pop()
        else:                        stack.append(c)
    return ''.join(stack)

# ── Tests ────────────────────────────────────────────────────────────────
ms = MinStack(); ms.push(-2); ms.push(0); ms.push(-3)
print(ms.getMin(), ms.top())                       # -3  -3
ms.pop(); print(ms.getMin())                       # -2

print(is_valid_parens("()[]{}" ))                  # True
print(is_valid_parens("([)]"))                     # False
print(longest_valid_parens(")()())"))              # 4
print(remove_duplicates("abbaca"))                 # "ca"


## 2.3 Queue — Deque Pattern & Implementation

### Mental Model
- Regular queue: FIFO using `collections.deque` with `appendleft`/`pop` or `append`/`popleft`
- Deque: double-ended — add/remove from both ends in O(1)

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 239  | Sliding Window Maximum | 🔴 |
| LC 641  | Design Circular Deque | 🟡 |
| LC 225  | Implement Stack using Queues | 🟢 |
| LC 232  | Implement Queue using Stacks | 🟢 |
| LC 862  | Shortest Subarray with Sum >= K | 🔴 |


In [ ]:
# ── Queue & Deque Templates ──────────────────────────────────────────────
from collections import deque

# Implement Queue using Two Stacks
class MyQueue:
    def __init__(self):
        self.in_stack = []
        self.out_stack = []
    def push(self, x): self.in_stack.append(x)
    def _transfer(self):
        if not self.out_stack:
            while self.in_stack: self.out_stack.append(self.in_stack.pop())
    def pop(self):   self._transfer(); return self.out_stack.pop()
    def peek(self):  self._transfer(); return self.out_stack[-1]
    def empty(self): return not self.in_stack and not self.out_stack

# Shortest subarray with sum >= k (monotonic deque on prefix sums)
def shortest_subarray_sum_k(nums, k):
    n = len(nums)
    prefix = [0] * (n + 1)
    for i in range(n): prefix[i+1] = prefix[i] + nums[i]
    dq = deque()    # stores indices of prefix, increasing order
    res = n + 1
    for i in range(n + 1):
        while dq and prefix[i] - prefix[dq[0]] >= k:
            res = min(res, i - dq.popleft())
        while dq and prefix[dq[-1]] >= prefix[i]:
            dq.pop()
        dq.append(i)
    return res if res <= n else -1

# ── Tests ────────────────────────────────────────────────────────────────
q = MyQueue(); q.push(1); q.push(2)
print(q.peek(), q.pop(), q.empty())   # 1 1 False
print(shortest_subarray_sum_k([2,-1,2], 3))   # 3


---
# 3. Trees
---

In [ ]:
# ── TreeNode definition (used throughout section 3) ──────────────────────
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val; self.left = left; self.right = right

def build_tree(vals):
    """Build tree from level-order list (None = missing node)."""
    if not vals or vals[0] is None: return None
    root = TreeNode(vals[0])
    q = [root]; i = 1
    while q and i < len(vals):
        node = q.pop(0)
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i]); q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i]); q.append(node.right)
        i += 1
    return root


## 3.1 Tree BFS (Level-order traversal)

### Mental Model
Use a queue. Process nodes level by level. BFS is the right tool whenever
you need "shortest path" or "level X" information in a tree.

### Complexity: O(n) time, O(w) space where w = max width

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 102 | Binary Tree Level Order | 🟡 |
| LC 103 | Zigzag Level Order | 🟡 |
| LC 199 | Right Side View | 🟡 |
| LC 111 | Min Depth | 🟢 |
| LC 117 | Populating Next Right Pointers II | 🟡 |


In [ ]:
# ── Tree BFS Templates ──────────────────────────────────────────────────
from collections import deque

# Level order traversal
def level_order(root):
    if not root: return []
    res = []; q = deque([root])
    while q:
        level = []
        for _ in range(len(q)):
            node = q.popleft()
            level.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        res.append(level)
    return res

# Right side view (last node at each level)
def right_side_view(root):
    res = []; q = deque([root]) if root else deque()
    while q:
        for i in range(len(q)):
            node = q.popleft()
            if i == len(q): res.append(node.val)   # last at this level
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        if q or res: res = res   # last node at level is already popped
    # cleaner:
    return res

def right_side_view_v2(root):
    res = []; q = deque([root]) if root else deque()
    while q:
        for _ in range(len(q) - 1):
            n = q.popleft()
            if n.left: q.append(n.left)
            if n.right: q.append(n.right)
        last = q.popleft()
        res.append(last.val)
        if last.left: q.append(last.left)
        if last.right: q.append(last.right)
    return res

# ── Tests ────────────────────────────────────────────────────────────────
root = build_tree([3,9,20,None,None,15,7])
print(level_order(root))        # [[3],[9,20],[15,7]]
print(right_side_view_v2(root)) # [3,20,7]


## 3.2 Tree DFS & Recursion

### Mental Model
Three traversals: pre-order (root→left→right), in-order (left→root→right), post-order (left→right→root).

**DFS state:** Think about what each recursive call must *return* and what it must *track* in the parent.

### Complexity: O(n) time, O(h) space (h = height)

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 104 | Max Depth | 🟢 |
| LC 543 | Diameter of Binary Tree | 🟢 |
| LC 236 | LCA of Binary Tree | 🟡 |
| LC 124 | Binary Tree Max Path Sum | 🔴 |
| LC 110 | Balanced Binary Tree | 🟢 |


In [ ]:
# ── Tree DFS Templates ──────────────────────────────────────────────────

# Max depth
def max_depth(root):
    if not root: return 0
    return 1 + max(max_depth(root.left), max_depth(root.right))

# Diameter (longest path between any two nodes)
def diameter_of_binary_tree(root):
    best = [0]
    def height(node):
        if not node: return 0
        L = height(node.left)
        R = height(node.right)
        best[0] = max(best[0], L + R)  # path through this node
        return 1 + max(L, R)
    height(root)
    return best[0]

# LCA — Lowest Common Ancestor
def lowest_common_ancestor(root, p, q):
    if not root or root == p or root == q: return root
    left  = lowest_common_ancestor(root.left, p, q)
    right = lowest_common_ancestor(root.right, p, q)
    # if found on both sides → current node is LCA
    return root if left and right else (left or right)

# Max path sum (path can start/end at any node)
def max_path_sum(root):
    best = [float('-inf')]
    def dfs(node):
        if not node: return 0
        L = max(dfs(node.left), 0)   # ignore negative branches
        R = max(dfs(node.right), 0)
        best[0] = max(best[0], node.val + L + R)
        return node.val + max(L, R)
    dfs(root)
    return best[0]

# ── Tests ────────────────────────────────────────────────────────────────
root = build_tree([1,2,3,4,5])
print(max_depth(root))                   # 3
print(diameter_of_binary_tree(root))     # 3 (path 4-2-1-3 or 5-2-1-3)

root2 = build_tree([-10,9,20,None,None,15,7])
print(max_path_sum(root2))               # 42 (15+20+7)


## 3.3 BST Operations

### Mental Model
BST property: `left.val < node.val < right.val` — use this to guide search in O(h) instead of O(n).
In-order traversal of a BST gives a sorted sequence.

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 98  | Validate BST | 🟡 |
| LC 230 | Kth Smallest in BST | 🟡 |
| LC 235 | LCA of BST | 🟢 |
| LC 108 | Sorted Array to BST | 🟢 |
| LC 450 | Delete Node in BST | 🟡 |


In [ ]:
# ── BST Templates ───────────────────────────────────────────────────────

# Validate BST (pass min/max bounds down)
def is_valid_bst(root, lo=float('-inf'), hi=float('inf')):
    if not root: return True
    if not (lo < root.val < hi): return False
    return (is_valid_bst(root.left, lo, root.val) and
            is_valid_bst(root.right, root.val, hi))

# Kth smallest (in-order traversal)
def kth_smallest(root, k):
    stack = []; node = root
    while stack or node:
        while node: stack.append(node); node = node.left
        node = stack.pop()
        k -= 1
        if k == 0: return node.val
        node = node.right

# LCA of BST (use BST property)
def lca_bst(root, p, q):
    while root:
        if p.val < root.val and q.val < root.val: root = root.left
        elif p.val > root.val and q.val > root.val: root = root.right
        else: return root

# Sorted array to balanced BST
def sorted_array_to_bst(nums):
    if not nums: return None
    mid = len(nums) // 2
    node = TreeNode(nums[mid])
    node.left  = sorted_array_to_bst(nums[:mid])
    node.right = sorted_array_to_bst(nums[mid+1:])
    return node

# ── Tests ────────────────────────────────────────────────────────────────
bst = build_tree([5,3,6,2,4,None,None,1])
print(is_valid_bst(bst))              # True
print(kth_smallest(bst, 3))          # 3


---
# 4. Graphs
---

## 4.1 Graph BFS & DFS

### Mental Model
- **BFS:** level-by-level, shortest path (unweighted), connected components
- **DFS:** explore fully before backtracking, cycle detection, topological sort, connected components

### Representation
- Adjacency list: `{node: [neighbours]}` — preferred for sparse graphs
- Adjacency matrix: `grid[i][j]` — used for 2D problems

### Complexity
| | BFS | DFS |
|--|-----|-----|
| Time  | O(V+E) | O(V+E) |
| Space | O(V)   | O(V) stack |

### Common Mistakes
- ❌ Not marking nodes visited before enqueuing (BFS) → duplicates
- ❌ Using recursion for very deep DFS → stack overflow (use iterative DFS)
- ❌ Forgetting to handle disconnected graphs — loop over ALL nodes as potential sources

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 200  | Number of Islands | 🟡 |
| LC 133  | Clone Graph | 🟡 |
| LC 127  | Word Ladder | 🔴 |
| LC 994  | Rotting Oranges | 🟡 |
| LC 695  | Max Area of Island | 🟡 |


In [ ]:
# ── Graph BFS & DFS Templates ────────────────────────────────────────────
from collections import deque

# Number of islands (BFS / DFS on grid)
def num_islands_bfs(grid):
    if not grid: return 0
    nr, nc = len(grid), len(grid[0])
    count = 0
    def bfs(r, c):
        q = deque([(r,c)]); grid[r][c] = '0'
        while q:
            r,c = q.popleft()
            for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr2,nc2 = r+dr, c+dc
                if 0<=nr2<nr and 0<=nc2<nc and grid[nr2][nc2]=='1':
                    grid[nr2][nc2]='0'; q.append((nr2,nc2))
    for r in range(nr):
        for c in range(nc):
            if grid[r][c]=='1': count+=1; bfs(r,c)
    return count

# Rotting oranges (multi-source BFS)
def oranges_rotting(grid):
    nr, nc = len(grid), len(grid[0])
    q = deque(); fresh = 0
    for r in range(nr):
        for c in range(nc):
            if grid[r][c]==2: q.append((r,c,0))
            elif grid[r][c]==1: fresh+=1
    minutes = 0
    while q:
        r,c,t = q.popleft()
        for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr2,nc2 = r+dr,c+dc
            if 0<=nr2<nr and 0<=nc2<nc and grid[nr2][nc2]==1:
                grid[nr2][nc2]=2; fresh-=1
                minutes=t+1; q.append((nr2,nc2,t+1))
    return minutes if fresh==0 else -1

# General graph BFS (adjacency list)
def bfs_graph(graph, start):
    visited = {start}; q = deque([start]); order = []
    while q:
        node = q.popleft(); order.append(node)
        for nb in graph[node]:
            if nb not in visited: visited.add(nb); q.append(nb)
    return order

# ── Tests ────────────────────────────────────────────────────────────────
import copy
grid = [["1","1","0","0","0"],["1","1","0","0","0"],
        ["0","0","1","0","0"],["0","0","0","1","1"]]
print(num_islands_bfs(copy.deepcopy(grid)))  # 3

rotten = [[2,1,1],[1,1,0],[0,1,1]]
print(oranges_rotting(rotten))  # 4


## 4.2 Topological Sort

### Mental Model
Order nodes in a DAG so all edges go from earlier to later (prerequisites before course).
Two approaches:
1. **Kahn's BFS:** in-degree count → start with 0-in-degree nodes
2. **DFS-based:** post-order DFS → reverse gives topo order

### Common Mistakes
- ❌ Not detecting cycles — if not all nodes processed, cycle exists
- ❌ Forgetting to initialise in-degree for all nodes including those with no incoming edges

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 207 | Course Schedule | 🟡 |
| LC 210 | Course Schedule II | 🟡 |
| LC 269 | Alien Dictionary | 🔴 |
| LC 310 | Minimum Height Trees | 🟡 |
| LC 1203 | Sort Items by Groups | 🔴 |


In [ ]:
# ── Topological Sort Templates ───────────────────────────────────────────
from collections import deque, defaultdict

# Kahn's algorithm (BFS-based)
def topo_sort_kahn(n, prerequisites):
    graph = defaultdict(list)
    in_degree = [0] * n
    for a, b in prerequisites:   # b → a (b must come before a)
        graph[b].append(a)
        in_degree[a] += 1
    q = deque(i for i in range(n) if in_degree[i] == 0)
    order = []
    while q:
        node = q.popleft(); order.append(node)
        for nb in graph[node]:
            in_degree[nb] -= 1
            if in_degree[nb] == 0: q.append(nb)
    return order if len(order) == n else []   # empty → cycle detected

# Can finish all courses?
def can_finish(num_courses, prerequisites):
    return len(topo_sort_kahn(num_courses, prerequisites)) == num_courses

# ── Tests ────────────────────────────────────────────────────────────────
print(can_finish(2, [[1,0]]))               # True
print(can_finish(2, [[1,0],[0,1]]))         # False (cycle)
print(topo_sort_kahn(4, [[1,0],[2,0],[3,1],[3,2]])) # [0,1,2,3] or similar


## 4.3 Shortest Path (Dijkstra's)

### Mental Model
Greedy BFS with a min-heap. Always process the unvisited node with smallest known distance.

**Use when:** Weighted graph, non-negative weights, single-source shortest path.

### Complexity: O((V + E) log V)

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 743 | Network Delay Time | 🟡 |
| LC 787 | Cheapest Flights K Stops | 🟡 |
| LC 1631 | Min Effort Path | 🟡 |
| LC 778 | Swim in Rising Water | 🔴 |
| LC 1514 | Path with Max Probability | 🟡 |


In [ ]:
# ── Dijkstra's Algorithm ────────────────────────────────────────────────
import heapq
from collections import defaultdict

def dijkstra(n, edges, source):
    """Returns shortest dist from source to all nodes. Nodes 1..n."""
    graph = defaultdict(list)
    for u, v, w in edges: graph[u].append((v, w))
    dist = [float('inf')] * (n + 1)
    dist[source] = 0
    heap = [(0, source)]   # (distance, node)
    while heap:
        d, u = heapq.heappop(heap)
        if d > dist[u]: continue     # stale entry
        for v, w in graph[u]:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                heapq.heappush(heap, (dist[v], v))
    return dist[1:]   # dist[1..n]

# Network delay time (max of all shortest paths from k)
def network_delay_time(times, n, k):
    dist = dijkstra(n, times, k)
    return max(dist) if max(dist) < float('inf') else -1

# ── Tests ────────────────────────────────────────────────────────────────
times = [[2,1,1],[2,3,1],[3,4,1]]
print(network_delay_time(times, 4, 2))   # 2


## 4.4 Cycle Detection & Connected Components (Union-Find)

### Mental Model
Union-Find (Disjoint Set Union) maintains a forest where each tree = one component.
`find(x)` with path compression: O(α) ≈ O(1). `union(x, y)` with rank: O(α).

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 684 | Redundant Connection | 🟡 |
| LC 547 | Number of Provinces | 🟡 |
| LC 399 | Evaluate Division | 🟡 |
| LC 323 | Number of Connected Components | 🟡 |
| LC 261 | Graph Valid Tree | 🟡 |


In [ ]:
# ── Union-Find Template ──────────────────────────────────────────────────

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank   = [0] * n
        self.count  = n    # number of components

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # path compression
        return self.parent[x]

    def union(self, x, y):
        px, py = self.find(x), self.find(y)
        if px == py: return False   # already connected → cycle!
        if self.rank[px] < self.rank[py]: px, py = py, px
        self.parent[py] = px
        if self.rank[px] == self.rank[py]: self.rank[px] += 1
        self.count -= 1
        return True

# Redundant connection
def find_redundant_connection(edges):
    uf = UnionFind(len(edges) + 1)
    for u, v in edges:
        if not uf.union(u, v): return [u, v]
    return []

# Number of provinces
def find_circle_num(is_connected):
    n = len(is_connected)
    uf = UnionFind(n)
    for i in range(n):
        for j in range(i+1, n):
            if is_connected[i][j]: uf.union(i, j)
    return uf.count

# ── Tests ────────────────────────────────────────────────────────────────
print(find_redundant_connection([[1,2],[1,3],[2,3]]))  # [2,3]
print(find_circle_num([[1,1,0],[1,1,0],[0,0,1]]))      # 2


---
# 5. Dynamic Programming
---

## 5.1 1D DP

### 5-Step DP Framework
1. **Define state:** What does `dp[i]` represent?
2. **Write recurrence:** How does `dp[i]` depend on smaller states?
3. **Base cases:** What are the smallest valid inputs?
4. **Fill order:** Left-to-right, right-to-left, or top-down with memo?
5. **Return value:** `dp[n]`? `max(dp)`? Something else?

### Practice Problems
| # | Problem | Difficulty | Pattern |
|---|---------|-----------|---------|
| LC 70  | Climbing Stairs | 🟢 | fibonacci |
| LC 322 | Coin Change | 🟡 | unbounded knapsack |
| LC 300 | LIS | 🟡 | patience sort |
| LC 139 | Word Break | 🟡 | string DP |
| LC 198 | House Robber | 🟢 | state machine |


In [ ]:
# ── 1D DP Templates ─────────────────────────────────────────────────────

# Climbing stairs (Fibonacci variant)
def climb_stairs(n):
    if n <= 2: return n
    a, b = 1, 2
    for _ in range(3, n+1): a, b = b, a+b
    return b

# House Robber (state machine: rob or skip)
def house_robber(nums):
    prev2 = prev1 = 0
    for x in nums:
        prev2, prev1 = prev1, max(prev1, prev2 + x)
    return prev1

# House Robber II (circular — solve twice)
def house_robber_ii(nums):
    def rob(arr):
        prev2 = prev1 = 0
        for x in arr:
            prev2, prev1 = prev1, max(prev1, prev2+x)
        return prev1
    n = len(nums)
    if n == 1: return nums[0]
    return max(rob(nums[:-1]), rob(nums[1:]))

# Word Break
def word_break(s, word_dict):
    n = len(s); word_set = set(word_dict)
    dp = [False] * (n+1); dp[0] = True
    for i in range(1, n+1):
        for j in range(i):
            if dp[j] and s[j:i] in word_set:
                dp[i] = True; break
    return dp[n]

# LIS — O(n log n) patience sort
from bisect import bisect_left
def length_of_LIS(nums):
    tails = []
    for x in nums:
        pos = bisect_left(tails, x)
        if pos == len(tails): tails.append(x)
        else:                 tails[pos] = x
    return len(tails)

# ── Tests ────────────────────────────────────────────────────────────────
print(climb_stairs(10))                          # 89
print(house_robber([2,7,9,3,1]))                # 12
print(house_robber_ii([2,3,2]))                 # 3
print(word_break("leetcode", ["leet","code"]))  # True
print(length_of_LIS([10,9,2,5,3,7,101,18]))    # 4


## 5.2 2D DP & Subsequences

### Practice Problems
| # | Problem | Difficulty | Pattern |
|---|---------|-----------|---------|
| LC 1143 | LCS | 🟡 | 2D DP |
| LC 72   | Edit Distance | 🔴 | 2D DP |
| LC 516  | Longest Palindromic Subsequence | 🟡 | interval DP |
| LC 62   | Unique Paths | 🟡 | grid DP |
| LC 221  | Maximal Square | 🟡 | grid DP |


In [ ]:
# ── 2D DP & Subsequences ────────────────────────────────────────────────

# Longest Common Subsequence
def lcs(text1, text2):
    m, n = len(text1), len(text2)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            if text1[i-1] == text2[j-1]: dp[i][j] = dp[i-1][j-1] + 1
            else:                         dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]

# Edit Distance
def edit_distance(word1, word2):
    m, n = len(word1), len(word2)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            if word1[i-1] == word2[j-1]: dp[i][j] = dp[i-1][j-1]
            else: dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[m][n]

# Longest Palindromic Subsequence (LCS of s and reverse)
def longest_palindromic_subseq(s):
    return lcs(s, s[::-1])

# Maximal Square
def maximal_square(matrix):
    if not matrix: return 0
    nr, nc = len(matrix), len(matrix[0])
    dp = [[0]*nc for _ in range(nr)]; best = 0
    for i in range(nr):
        for j in range(nc):
            if matrix[i][j] == '1':
                dp[i][j] = (1 if i==0 or j==0 else
                            min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1]) + 1)
                best = max(best, dp[i][j])
    return best * best

# ── Tests ────────────────────────────────────────────────────────────────
print(lcs("abcde", "ace"))              # 3
print(edit_distance("horse","ros"))     # 3
print(longest_palindromic_subseq("bbbab"))  # 4
m = [["1","0","1","0","0"],["1","0","1","1","1"],["1","1","1","1","1"],["1","0","0","1","0"]]
print(maximal_square(m))                # 4


## 5.3 Knapsack Patterns

### Types
| Type | Constraint | Classic Problem |
|------|-----------|----------------|
| 0/1 Knapsack | each item used at most once | LC 416 Partition Equal Subset |
| Unbounded | unlimited uses of each item | LC 322 Coin Change |
| Bounded | limited count of each item | — |

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 416 | Partition Equal Subset Sum | 🟡 |
| LC 494 | Target Sum | 🟡 |
| LC 474 | Ones and Zeroes | 🟡 |
| LC 518 | Coin Change 2 | 🟡 |
| LC 1049| Last Stone Weight II | 🟡 |


In [ ]:
# ── Knapsack Patterns ────────────────────────────────────────────────────

# 0/1 Knapsack — Partition Equal Subset Sum
def can_partition(nums):
    total = sum(nums)
    if total % 2: return False
    target = total // 2
    dp = {0}   # reachable sums (set-based knapsack)
    for x in nums:
        dp |= {s + x for s in dp}
    return target in dp

# Coin Change 2 — count ways (unbounded knapsack, order doesn't matter)
def coin_change_2(amount, coins):
    dp = [0] * (amount + 1); dp[0] = 1
    for coin in coins:                  # outer: items
        for a in range(coin, amount+1): # inner: amounts
            dp[a] += dp[a-coin]
    return dp[amount]

# Target Sum — count assignments (+/-) to reach target
def find_target_sum_ways(nums, target):
    # P - N = target, P + N = total  →  P = (total+target)/2
    total = sum(nums)
    if (total + target) % 2 or abs(target) > total: return 0
    cap = (total + target) // 2
    dp = [0] * (cap + 1); dp[0] = 1
    for x in nums:
        for j in range(cap, x-1, -1):   # 0/1: iterate backwards
            dp[j] += dp[j-x]
    return dp[cap]

# ── Tests ────────────────────────────────────────────────────────────────
print(can_partition([1,5,11,5]))    # True (5+6 vs 11 — wait: 1+5+5=11 yes)
print(coin_change_2(5, [1,2,5]))   # 4
print(find_target_sum_ways([1,1,1,1,1], 3)) # 5


---
# 6. Trie (Prefix Tree)
---

## Trie

### Mental Model
A tree where each node represents a character. Paths from root to a leaf spell a word.
Perfect for prefix queries, autocomplete, and dictionary-based word problems.

### Complexity
| Operation | Time | Space |
|-----------|------|-------|
| Insert    | O(L) | O(L·26) |
| Search    | O(L) | — |
| Prefix    | O(L) | — |
(L = word length)

### Common Mistakes
- ❌ Forgetting to mark `is_end=True` at the last character of a word
- ❌ Searching vs prefix check confusion — search requires `is_end`, prefix doesn't
- ❌ Not initialising `children` as a dict vs array (dict is simpler, array faster)

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 208  | Implement Trie | 🟡 |
| LC 212  | Word Search II | 🔴 |
| LC 211  | Design Add and Search | 🟡 |
| LC 421  | Maximum XOR of Two Numbers | 🟡 |
| LC 745  | Prefix and Suffix Search | 🔴 |


In [ ]:
# ── Trie Implementation ─────────────────────────────────────────────────

class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for c in word:
            if c not in node.children:
                node.children[c] = TrieNode()
            node = node.children[c]
        node.is_end = True

    def search(self, word):
        node = self.root
        for c in word:
            if c not in node.children: return False
            node = node.children[c]
        return node.is_end

    def starts_with(self, prefix):
        node = self.root
        for c in prefix:
            if c not in node.children: return False
            node = node.children[c]
        return True

# Design Add and Search Words (with '.' wildcard)
class WordDictionary:
    def __init__(self): self.root = TrieNode()

    def add_word(self, word):
        node = self.root
        for c in word:
            if c not in node.children: node.children[c] = TrieNode()
            node = node.children[c]
        node.is_end = True

    def search(self, word):
        def dfs(node, i):
            if i == len(word): return node.is_end
            c = word[i]
            if c == '.':
                return any(dfs(child, i+1) for child in node.children.values())
            if c not in node.children: return False
            return dfs(node.children[c], i+1)
        return dfs(self.root, 0)

# ── Tests ────────────────────────────────────────────────────────────────
t = Trie()
t.insert("apple"); t.insert("app")
print(t.search("apple"))          # True
print(t.search("ap"))             # False
print(t.starts_with("app"))       # True

wd = WordDictionary()
wd.add_word("bad"); wd.add_word("dad"); wd.add_word("mad")
print(wd.search("pad"))   # False
print(wd.search(".ad"))   # True
print(wd.search("b.."))   # True


---
# 7. Heap (Priority Queue)
---

## Heap

### Mental Model
Python's `heapq` is a **min-heap**. For max-heap, negate values.
Key insight: heap gives you the min/max in O(1), and insert/extract in O(log n).

### When to use
- K largest/smallest elements
- Merge K sorted lists/arrays
- Top K frequent elements
- Scheduling (always process the job with smallest deadline)
- Median of a stream (two heaps)

### Complexity
| Operation | Time |
|-----------|------|
| Push      | O(log n) |
| Pop       | O(log n) |
| Peek      | O(1) |
| Heapify   | O(n) |

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 215  | Kth Largest Element | 🟡 |
| LC 347  | Top K Frequent Elements | 🟡 |
| LC 295  | Find Median from Data Stream | 🔴 |
| LC 23   | Merge K Sorted Lists | 🔴 |
| LC 621  | Task Scheduler | 🟡 |


In [ ]:
# ── Heap Templates ──────────────────────────────────────────────────────
import heapq
from collections import Counter

# Kth largest element (min-heap of size k)
def find_kth_largest(nums, k):
    heap = []
    for x in nums:
        heapq.heappush(heap, x)
        if len(heap) > k: heapq.heappop(heap)
    return heap[0]

# Top K frequent elements
def top_k_frequent(nums, k):
    count = Counter(nums)
    # bucket sort: index=frequency, value=list of nums
    buckets = [[] for _ in range(len(nums)+1)]
    for num, freq in count.items(): buckets[freq].append(num)
    res = []
    for i in range(len(buckets)-1, -1, -1):
        res.extend(buckets[i])
        if len(res) >= k: return res[:k]
    return res[:k]

# Median from data stream (two heaps)
class MedianFinder:
    def __init__(self):
        self.lo = []   # max-heap (negate values) — lower half
        self.hi = []   # min-heap — upper half

    def add_num(self, num):
        heapq.heappush(self.lo, -num)
        # balance: lo max must be <= hi min
        if self.hi and -self.lo[0] > self.hi[0]:
            heapq.heappush(self.hi, -heapq.heappop(self.lo))
        # size balance: lo can be 1 larger
        if len(self.lo) > len(self.hi) + 1:
            heapq.heappush(self.hi, -heapq.heappop(self.lo))
        elif len(self.hi) > len(self.lo):
            heapq.heappush(self.lo, -heapq.heappop(self.hi))

    def find_median(self):
        if len(self.lo) > len(self.hi): return -self.lo[0]
        return (-self.lo[0] + self.hi[0]) / 2

# Task Scheduler
def least_interval(tasks, n):
    count = Counter(tasks)
    max_count = max(count.values())
    num_max = sum(1 for c in count.values() if c == max_count)
    return max(len(tasks), (max_count - 1) * (n + 1) + num_max)

# ── Tests ────────────────────────────────────────────────────────────────
print(find_kth_largest([3,2,1,5,6,4], 2))    # 5
print(top_k_frequent([1,1,1,2,2,3], 2))       # [1,2]

mf = MedianFinder()
for x in [1,2,3]: mf.add_num(x)
print(mf.find_median())    # 2.0

print(least_interval(["A","A","A","B","B","B"], 2))  # 8


---
# 8. Linked List
---

## Linked List

### Core Tricks
1. **Dummy head:** Create a sentinel node before the head to avoid edge cases on insertion/deletion
2. **Fast & Slow pointers:** Find middle, detect cycle, find kth from end
3. **Reverse in segments:** Reverse entire list, or segments (k-groups)

### Common Mistakes
- ❌ Not saving `next` before overwriting the pointer (`.next = next_node` then `lost = node.next`)
- ❌ Off-by-one in finding kth from end: advance fast k steps first, then move both
- ❌ Forgetting to null-terminate the new tail after splitting/reversing

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 206  | Reverse Linked List | 🟢 |
| LC 141  | Linked List Cycle | 🟢 |
| LC 142  | Linked List Cycle II | 🟡 |
| LC 21   | Merge Two Sorted Lists | 🟢 |
| LC 25   | Reverse Nodes in K-Group | 🔴 |
| LC 19   | Remove Nth Node From End | 🟡 |


In [ ]:
# ── Linked List Templates ────────────────────────────────────────────────

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val; self.next = next

def to_list(head):
    res = []
    while head: res.append(head.val); head = head.next
    return res

def from_list(vals):
    dummy = ListNode(); cur = dummy
    for v in vals: cur.next = ListNode(v); cur = cur.next
    return dummy.next

# Reverse Linked List (iterative)
def reverse_list(head):
    prev = None; cur = head
    while cur:
        nxt = cur.next
        cur.next = prev
        prev = cur; cur = nxt
    return prev

# Detect cycle (Floyd's)
def has_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next; fast = fast.next.next
        if slow == fast: return True
    return False

# Find cycle start
def detect_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next; fast = fast.next.next
        if slow == fast:
            slow = head
            while slow != fast: slow = slow.next; fast = fast.next
            return slow
    return None

# Remove Nth from end (two pointers, one pass)
def remove_nth_from_end(head, n):
    dummy = ListNode(0, head)
    fast = slow = dummy
    for _ in range(n+1): fast = fast.next
    while fast:
        fast = fast.next; slow = slow.next
    slow.next = slow.next.next
    return dummy.next

# Merge Two Sorted Lists
def merge_two_lists(l1, l2):
    dummy = cur = ListNode()
    while l1 and l2:
        if l1.val <= l2.val: cur.next = l1; l1 = l1.next
        else:                cur.next = l2; l2 = l2.next
        cur = cur.next
    cur.next = l1 or l2
    return dummy.next

# ── Tests ────────────────────────────────────────────────────────────────
lst = from_list([1,2,3,4,5])
print(to_list(reverse_list(lst)))              # [5,4,3,2,1]

l1 = from_list([1,2,4]); l2 = from_list([1,3,4])
print(to_list(merge_two_lists(l1,l2)))         # [1,1,2,3,4,4]

lst2 = from_list([1,2,3,4,5])
print(to_list(remove_nth_from_end(lst2, 2)))   # [1,2,3,5]


---
# 9. Matrix Traversal
---

## Matrix Traversal Patterns

### Patterns
1. **Spiral order** — directional boundaries shrinking inward
2. **Diagonal traversal** — index math tricks
3. **Rotate matrix** — transpose then reverse rows
4. **Search in sorted matrix** — binary search or staircase search

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 54  | Spiral Matrix | 🟡 |
| LC 48  | Rotate Image | 🟡 |
| LC 74  | Search a 2D Matrix | 🟡 |
| LC 240 | Search a 2D Matrix II | 🟡 |
| LC 73  | Set Matrix Zeroes | 🟡 |


In [ ]:
# ── Matrix Traversal Templates ──────────────────────────────────────────

# Spiral Matrix
def spiral_order(matrix):
    res = []
    top, bottom, left, right = 0, len(matrix)-1, 0, len(matrix[0])-1
    while top <= bottom and left <= right:
        for c in range(left, right+1): res.append(matrix[top][c])
        top += 1
        for r in range(top, bottom+1): res.append(matrix[r][right])
        right -= 1
        if top <= bottom:
            for c in range(right, left-1, -1): res.append(matrix[bottom][c])
            bottom -= 1
        if left <= right:
            for r in range(bottom, top-1, -1): res.append(matrix[r][left])
            left += 1
    return res

# Rotate Image 90° clockwise — transpose then reverse each row
def rotate(matrix):
    n = len(matrix)
    # Transpose
    for i in range(n):
        for j in range(i+1, n):
            matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]
    # Reverse each row
    for row in matrix: row.reverse()

# Search 2D Matrix II (staircase: start top-right)
def search_matrix(matrix, target):
    r, c = 0, len(matrix[0]) - 1
    while r < len(matrix) and c >= 0:
        if   matrix[r][c] == target: return True
        elif matrix[r][c] > target:  c -= 1
        else:                         r += 1
    return False

# Set Matrix Zeroes (O(1) space using first row/col as flags)
def set_zeroes(matrix):
    nr, nc = len(matrix), len(matrix[0])
    first_row_zero = any(matrix[0][j]==0 for j in range(nc))
    first_col_zero = any(matrix[i][0]==0 for i in range(nr))
    for i in range(1, nr):
        for j in range(1, nc):
            if matrix[i][j]==0: matrix[i][0]=matrix[0][j]=0
    for i in range(1, nr):
        for j in range(1, nc):
            if matrix[i][0]==0 or matrix[0][j]==0: matrix[i][j]=0
    if first_row_zero:
        for j in range(nc): matrix[0][j]=0
    if first_col_zero:
        for i in range(nr): matrix[i][0]=0

# ── Tests ────────────────────────────────────────────────────────────────
print(spiral_order([[1,2,3],[4,5,6],[7,8,9]]))  # [1,2,3,6,9,8,7,4,5]

m = [[1,2,3],[4,5,6],[7,8,9]]; rotate(m); print(m)  # rotated 90°
print(search_matrix([[1,4,7,11],[2,5,8,12],[3,6,9,16],[10,13,14,17]], 5))  # True


---
# 10. Line Sweep & Intervals
---

## Line Sweep

### Mental Model
Convert interval events into point events at start/end, sort them, then sweep a pointer left-to-right
tracking the "active" count (overlaps, coverage, etc).

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 253  | Meeting Rooms II | 🟡 |
| LC 218  | Skyline Problem | 🔴 |
| LC 56   | Merge Intervals | 🟡 |
| LC 1272 | Remove Covered Intervals | 🟡 |
| LC 57   | Insert Interval | 🟡 |


In [ ]:
# ── Line Sweep Templates ─────────────────────────────────────────────────

# Meeting Rooms II (min rooms needed = max simultaneous meetings)
def min_meeting_rooms(intervals):
    events = []
    for s, e in intervals:
        events.append((s, 1))    # meeting starts
        events.append((e, -1))   # meeting ends
    events.sort()
    rooms = cur = 0
    for _, delta in events:
        cur += delta
        rooms = max(rooms, cur)
    return rooms

# Insert Interval
def insert_interval(intervals, new_interval):
    res = []; i = 0; n = len(intervals)
    # intervals before new (no overlap)
    while i < n and intervals[i][1] < new_interval[0]:
        res.append(intervals[i]); i += 1
    # merge overlapping
    while i < n and intervals[i][0] <= new_interval[1]:
        new_interval[0] = min(new_interval[0], intervals[i][0])
        new_interval[1] = max(new_interval[1], intervals[i][1])
        i += 1
    res.append(new_interval)
    # intervals after new
    while i < n: res.append(intervals[i]); i += 1
    return res

# ── Tests ────────────────────────────────────────────────────────────────
print(min_meeting_rooms([[0,30],[5,10],[15,20]]))    # 2
print(insert_interval([[1,3],[6,9]], [2,5]))         # [[1,5],[6,9]]


---
# 11. Hashing (Deep Dive)
---

## Hashing Patterns

### Use cases
- O(1) membership check
- Frequency counting
- Two Sum pattern (complement lookup)
- Grouping by canonical form
- Prefix sum + hash for subarray problems

### Practice Problems
| # | Problem | Difficulty |
|---|---------|-----------|
| LC 1    | Two Sum | 🟢 |
| LC 49   | Group Anagrams | 🟡 |
| LC 128  | Longest Consecutive Sequence | 🟡 |
| LC 560  | Subarray Sum = K | 🟡 |
| LC 380  | Insert Delete GetRandom O(1) | 🟡 |


In [ ]:
# ── Hashing Templates ────────────────────────────────────────────────────
from collections import defaultdict, Counter
import random

# Longest Consecutive Sequence (O(n) using set)
def longest_consecutive(nums):
    num_set = set(nums); best = 0
    for n in num_set:
        if n - 1 not in num_set:  # start of a sequence
            cur = n; length = 1
            while cur + 1 in num_set: cur += 1; length += 1
            best = max(best, length)
    return best

# RandomizedSet: insert, remove, getRandom all O(1)
class RandomizedSet:
    def __init__(self):
        self.idx_map = {}   # val → index in list
        self.vals    = []

    def insert(self, val):
        if val in self.idx_map: return False
        self.idx_map[val] = len(self.vals)
        self.vals.append(val)
        return True

    def remove(self, val):
        if val not in self.idx_map: return False
        # swap with last element
        last = self.vals[-1]
        i = self.idx_map[val]
        self.vals[i] = last
        self.idx_map[last] = i
        self.vals.pop(); del self.idx_map[val]
        return True

    def get_random(self):
        return random.choice(self.vals)

# ── Tests ────────────────────────────────────────────────────────────────
print(longest_consecutive([100,4,200,1,3,2]))   # 4

rs = RandomizedSet()
print(rs.insert(1), rs.insert(2), rs.remove(1))  # True True True


---
# 12. Quick Reference — Complexity Cheat Sheet
---

## Algorithm → Complexity Map

| Algorithm | Time | Space | Notes |
|-----------|------|-------|-------|
| Binary Search | O(log n) | O(1) | sorted or monotone |
| Two Pointers | O(n) | O(1) | sorted or maintained invariant |
| Sliding Window | O(n) | O(k) | k = window size |
| Prefix Sum | O(n) build, O(1) query | O(n) | cumulative |
| Sorting | O(n log n) | O(log n) | Python timsort |
| BFS/DFS (graph) | O(V+E) | O(V) | V=nodes, E=edges |
| Dijkstra | O((V+E) log V) | O(V) | non-negative weights |
| Union-Find | O(α) ≈ O(1) | O(V) | with path compression + rank |
| Topo Sort | O(V+E) | O(V) | Kahn's BFS |
| Heap push/pop | O(log n) | O(n) | |
| Trie insert/search | O(L) | O(L·σ) | L=length, σ=alphabet |
| 1D DP | O(n) | O(n) or O(1) | depends on recurrence |
| 2D DP | O(n²) | O(n²) or O(n) | space can be optimised |
| Monotonic Stack | O(n) | O(n) | each element pushed/popped once |
| Backtracking | O(branch^depth) | O(depth) | exponential worst case |
| Kadane's | O(n) | O(1) | max subarray |

## Recognise the Pattern Fast

| Problem says | Think |
|-------------|-------|
| "sorted array + pair" | Two pointers |
| "longest/shortest subarray with property" | Sliding window |
| "range sum/query" | Prefix sum |
| "max/min K elements" | Heap |
| "next greater/smaller" | Monotonic stack |
| "valid brackets" | Stack |
| "connected components" | BFS/DFS or Union-Find |
| "prerequisites/ordering" | Topological sort |
| "shortest path weighted" | Dijkstra |
| "prefix matching, autocomplete" | Trie |
| "overlapping subproblems" | DP |
| "all combinations/permutations" | Backtracking |

## Interview Communication Checklist
1. ✅ Clarify constraints (size, range, edge cases) before coding
2. ✅ State brute force first with its complexity
3. ✅ Explain the optimisation idea before coding it
4. ✅ Code with clean variable names and comments
5. ✅ Walk through a test case manually after coding
6. ✅ State final time and space complexity unprompted
7. ✅ Discuss trade-offs if multiple approaches exist


---
## Practice Schedule (2-week plan before interview)

| Day | Topics | LeetCode Goals |
|-----|--------|---------------|
| 1-2 | Two Ptrs, Sliding Window, Prefix Sum | LC 3, 209, 560, 11, 15 |
| 3-4 | Sorting, Kadane's, Matrix DP | LC 56, 53, 152, 62, 221 |
| 5-6 | Stack (mono + parenthesis), Heap | LC 739, 84, 215, 295 |
| 7   | Trees: BFS, DFS, BST | LC 102, 543, 236, 124 |
| 8-9 | Graphs: BFS, DFS, Union-Find, Topo | LC 200, 207, 210, 684 |
| 10-11| DP: 1D, 2D, Knapsack | LC 322, 300, 1143, 416 |
| 12  | Trie, Linked List | LC 208, 206, 25, 23 |
| 13  | Line sweep, Hashing deep dive | LC 253, 128, 380 |
| 14  | Mock interview simulation (2 problems in 60 min) | Agoda-style |
